# 02 — Optional anomaly comparison (Isolation Forest)

Fit on dev, evaluated on test at the same 2,954-alert budget as the rules. Features: log1p(amount) + one-hot type only. Excluded: labels, balances, IDs, future aggregates.
Result on the sample: IF recall ~13% vs rules 32% — kept rules as primary because ops needs explainable reasons.

In [ ]:
import duckdb, numpy as np, pandas as pd
from pathlib import Path
from sklearn.ensemble import IsolationForest

con = duckdb.connect(str(Path('..') / 'data' / 'processed' / 'fraudops.duckdb'))
train = con.execute('SELECT amount, transaction_type FROM fact_transactions WHERE simulation_step <= 446').fetchdf()
train['log_amt'] = np.log1p(train['amount'])
X_train = pd.get_dummies(train[['log_amt', 'transaction_type']], columns=['transaction_type'], drop_first=True)
clf = IsolationForest(contamination=0.02, random_state=42).fit(X_train)
test = con.execute('SELECT transaction_id, amount, transaction_type, is_fraud FROM fact_transactions WHERE simulation_step BETWEEN 595 AND 744').fetchdf()
test['log_amt'] = np.log1p(test['amount'])
X_test = pd.get_dummies(test[['log_amt', 'transaction_type']], columns=['transaction_type'], drop_first=True)
for c in X_train.columns:
    X_test[c] = X_test.get(c, 0)
X_test = X_test[X_train.columns]
test['score'] = -clf.decision_function(X_test)
top = test.sort_values('score', ascending=False).head(2954)
tp = int((top.is_fraud == 1).sum())
print(f'IF top-2954: tp={tp} recall={tp / 177:.3f} vs rules tp=57 recall=0.322')